# RadHarmony: Evaluating a Foundation Model - Part 3




> **Please select `RadHarmony-(chexagent)` as the Python kernel.**

In [ ]:
import os
import pandas as pd

GPU_ID = 2
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
DEVICE = "cuda"

### Try it yourself: a different VQA model

Swap the recipe for **CheXagent-2** (`make_chexagent_vqa`), a CXR-specialist VLM.
The harness, dataset, prompt handling, and metrics stay put; only the two lines that
build the model change.

In [ ]:
from radharmony.evaluator import VQAEvaluator
from radharmony.dataset import VQARadDataset
from radharmony.evaluator.backbones import make_chexagent_vqa

VQA_RAD_DIR = "/mnt/NAS4/datasets/external/VQA-RAD/VQA_RAD Image Folder"

# CheXagent-2 instead of MedGemma; returns (transform, answerer) like the MedGemma recipe.
transform, answerer = make_chexagent_vqa(device=DEVICE)

vqa_ds = VQARadDataset(base_image_dir=VQA_RAD_DIR, transform=transform, cache_dir=None)

# FILL IN VQAEvaluator's first (positional) argument.
#   docs: https://f10409.github.io/RadHarmony/evaluator/vqa.html
ev = VQAEvaluator(answerer, dataset=vqa_ds, output_dir="outputs/vqa_chexagent")
ev.evaluate()

### Try it yourself: a different report generator

`make_chexagent_generator` (CheXagent-2) and `make_maira2_generator` (MAIRA-2) both
return `(transform, generator)` and drop straight into `ReportGenerationEvaluator`.
Swap the recipe; keep the MIMIC-CXR dataset, `findings` scoring, and the
RadGraph + BLEU metrics.

In [ ]:
from radharmony.evaluator.backbones import make_chexagent_generator
from radharmony.dataset import MIMICCXRDataset
from radharmony.evaluator import ReportGenerationEvaluator

MIMIC_IMAGE_DIR = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/files/"
MIMIC_REPORT_CSV = "/mnt/NAS4/datasets/external/MIMIC-CXR-V2-AWS/cxr-study-list.csv.gz"

# from radharmony.evaluator.backbones import make_maira2_generator  # MAIRA-2 instead
transform, generator = make_chexagent_generator(device=DEVICE)

df_harmonized = pd.read_parquet("df_mimic_harmonized.parquet")
gen_ds = MIMICCXRDataset(
    base_image_dir=MIMIC_IMAGE_DIR,
    report_csv_path=MIMIC_REPORT_CSV,
    transform=transform,
    output_report=True,
    cache_dir=None,
    harmonized_df=df_harmonized,
)

# FILL IN ref_section and metrics.
#   docs: https://f10409.github.io/RadHarmony/evaluator/report_generation.html
ev = ReportGenerationEvaluator(
    generator,
    dataset=gen_ds,
    ref_section="findings",  # which report section to score?
    metrics=["radgraph"],  # n-gram overlap + RadGraph F1
    max_samples=1000,  # small cap for the demo; remove for a full run
    output_dir="outputs/chexagent_report",
)
ev.evaluate()